# Did streaming change what a hit sounds like?

Final project for Data Acquisition and Management.
Julian Aldana, May 8 2026

---

## The question

Music journalists keep claiming that the rise of streaming services rewired what a popular song sounds
like. The argument goes that hits got shorter (because Spotify and Apple Music only pay royalties after
roughly 30 seconds of listening), that they got louder, less acoustic, more "front-loaded" with hooks,
and that explicit content exploded once labels could bypass radio gatekeepers. Some of those claims
are easy to find in op-eds. The harder question is whether the actual data backs them up.

So my research question is: comparing popular tracks from the album-and-radio era (1990 to 2009) to
the streaming era (2010 to 2020), are streaming-era hits measurably shorter, louder, less acoustic,
more danceable, and more often explicit, or are those claims industry folklore that the data doesn't
really support?

This notebook walks through how I built the dataset and what the numbers actually say. The short
version is that the real answer is somewhere in the middle. Three of the claims hold up clearly,
two do not, and there is one finding I wasn't expecting.

## Motivation

Two reasons I picked this. First, I'm into music and I wanted to spend a month on something I'd
actually be curious to read. Second, there is a real psychological angle here. Spotify's
valence score is literally a model of how positive or happy a song sounds, so a question like "did
pop music get sadder?" stops being pure vibes and becomes something that can be analyzed.

## Workflow / Steps

1. Acquire. Spotify track and audio-feature CSVs from Kaggle, plus a Billboard year-end Hot 100
   list scraped from Wikipedia for 1990 to 2020. Two different source types, relational and scraped.
2. Wrangle. Load everything into a normalized SQLite database, parse the stringified-list columns,
   match Billboard rows to Spotify track IDs.
3. Analyze. Yearly trends, era comparisons, bootstrap confidence intervals, genre breakdown.
4. Present. Plots and conclusions in this notebook.


## Section 1. Setup

A set of imports. I'm using pandas and numpy for data, matplotlib plus seaborn for plots, and the
built-in sqlite3 module for the database. I tried SQLAlchemy first but the built-in module is honestly
fine for this and it's one less thing for whoever runs this notebook to install.


In [ ]:
import os
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

# the database lives next to this notebook
DB = "music.db"
conn = sqlite3.connect(DB)

print("connected to", DB)


## Section 2. What's in the database?

Quick sanity check that all my tables loaded. If something is zero here, the load script didn't
work right and the rest of the notebook is going to be sad.


In [ ]:
tables = ["artists", "tracks", "audio_features", "track_artists",
          "genres", "artist_genres", "billboard_hits"]

print("Row counts in music.db:")
for t in tables:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t:18s}  {n:>12,}")

n_matched = conn.execute(
    "SELECT COUNT(*) FROM billboard_hits WHERE track_id IS NOT NULL"
).fetchone()[0]
print(f"\nbillboard_hits with a matched spotify track_id: {n_matched:,}")


The spotify dataset has about 580k tracks and 1.16M artists, which is way bigger than what I
actually need. The match rate on Billboard hits is around 84 percent, which is good enough. The
16 percent that didn't match are mostly tracks where the title contains punctuation that Wikipedia
and Spotify disagree about (things like "I'll" versus "I’ll"), or where Spotify has a
different version of the song than the chart edit.


## Section 3. Building the analysis frame

The actual question is about Billboard hits, meaning songs that the public actually bought or
streamed at chart-topping levels in a given year. Spotify's own popularity score is a fine proxy
but it reflects current listening, not historical chart performance, so I'd rather use Billboard.

I pull every matched Billboard hit, attach the spotify duration and loudness and audio features,
add a duration_s column, and tag each row with its era (album for 1990 to 2009, streaming for 2010
to 2020). This is the data transformation step from the rubric. The release_date column gets
turned into a year, the year gets turned into an era category, and duration_ms gets turned into
duration_s for plots that don't have to read like spec sheets.


In [ ]:
sql = '''
SELECT
    bb.bb_year       AS year,
    bb.bb_rank       AS rank,
    bb.bb_title      AS bb_title,
    bb.bb_artist     AS bb_artist,
    t.title          AS spotify_title,
    t.duration_ms,
    t.explicit,
    t.popularity,
    af.danceability,
    af.energy,
    af.loudness,
    af.acousticness,
    af.instrumentalness,
    af.speechiness,
    af.valence,
    af.tempo
FROM billboard_hits bb
JOIN tracks         t  ON t.track_id  = bb.track_id
JOIN audio_features af ON af.track_id = bb.track_id
WHERE bb.track_id IS NOT NULL
'''
hits = pd.read_sql(sql, conn)

# transformations
hits["duration_s"] = hits["duration_ms"] / 1000.0

def era_of(y):
    if 1990 <= y <= 2009:
        return "album"
    if 2010 <= y <= 2020:
        return "streaming"
    return "other"

hits["era"] = hits["year"].apply(era_of)

print(f"matched hits: {len(hits):,}")
print(hits["era"].value_counts())
hits.head()


## Section 4. Validation: do we have enough data per year?

Before I trust any era comparison I want to make sure I have a reasonable number of matched hits
in every year. If 2018 only has 5 matches and 1995 has 95, the era medians will be skewed.
This is the descriptive validation graphic for the rubric. It tells me the dataset is fit for
purpose before I go drawing any conclusions from it.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
hits.groupby("year").size().plot(kind="bar", ax=ax, color="#4C72B0")
# the dashed line marks where the streaming era starts
ax.axvline(x=20 - 0.5, color="red", ls="--", lw=1, label="streaming era starts (2010)")
ax.set_title("Matched Billboard hits per year (1990 to 2020)")
ax.set_xlabel("year")
ax.set_ylabel("# hits with audio features")
ax.legend()
plt.tight_layout()
plt.show()

per_year = hits.groupby("year").size()
print(f"min hits per year: {per_year.min()}    max: {per_year.max()}    mean: {per_year.mean():.0f}")


Roughly 70 to 90 matched hits per year, with no dramatic gaps. That's plenty.

## Section 5. Descriptive stats by era

This is the descriptive analysis the rubric asks for. Means, standard deviations, and quartiles
for each audio feature, split by era. Reading the median row for duration_s gives the headline
already (about 250 seconds in the album era versus about 219 in streaming), but the rest of the
table is here so I can sanity check that nothing is wildly weird (no negative durations, no
loudness scores outside Spotify's expected range, and so on).


In [ ]:
features = ["duration_s", "loudness", "acousticness", "danceability",
            "energy", "valence", "tempo"]

desc = hits.groupby("era")[features].describe().T
desc.round(2)


The key thing to eyeball here is the median duration row, around 250 seconds in the album era
versus around 219 in streaming. That's about 30 seconds shorter, which is a big change, and
it's not just the mean being pulled around by outliers. The median moved by basically the same
amount, which means the whole distribution shifted, not just a few extreme values.

## Section 6. Yearly trend plots

Six time series plotted as yearly medians, with a red dashed line at 2010 to mark the era
boundary. I picked this layout (one panel per audio feature, all sharing the same year axis)
because the question I most wanted to answer here is whether the change is gradual drift across
the whole period or a sharp break around the streaming transition. If it's gradual, the era
comparison later in the notebook is mostly a re-statement of long-running trends. If it's a
break around 2010, there is something specific to streaming worth talking about.


In [ ]:
yearly = hits.groupby("year")[features].median().reset_index()

fig, axes = plt.subplots(3, 2, figsize=(13, 10))
plot_specs = [
    ("duration_s",   "Median duration (seconds)",          axes[0,0]),
    ("loudness",     "Median loudness (dB, higher = louder)", axes[0,1]),
    ("acousticness", "Median acousticness (0-1)",          axes[1,0]),
    ("danceability", "Median danceability (0-1)",          axes[1,1]),
    ("energy",       "Median energy (0-1)",                axes[2,0]),
    ("valence",      "Median valence / positivity (0-1)",  axes[2,1]),
]
for col, label, ax in plot_specs:
    ax.plot(yearly["year"], yearly[col], marker="o", lw=1.5, color="#4C72B0")
    ax.axvline(x=2010, color="red", ls="--", lw=1, alpha=0.7)
    ax.set_title(label)
    ax.set_xlabel("year")
plt.tight_layout()
plt.show()


Eyeballing the panels, duration and valence both bend down sharply right around 2010. Loudness
has been trending up since the mid-90s. The "loudness wars" predate streaming, but the trend
keeps going. Acousticness and danceability look basically flat with year-to-year noise, which is
already an early hint that those parts of the streaming-era story might not survive a stats test.
Energy floats around without an obvious break.

## Section 7. Era boxplots

Same data, different angle. I plot the full distribution per era instead of just the median, so
I can see whether the overall shape of the audio profile is shifting or just the center. I picked
boxplots over histograms because side-by-side histograms with two eras get visually crowded fast,
whereas boxplots make it easy to see whether the box itself moved or whether the difference is
mostly in the tails.


In [ ]:
plot_features = ["duration_s", "loudness", "acousticness",
                 "energy", "valence", "tempo"]
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for i, col in enumerate(plot_features):
    ax = axes[i // 3, i % 3]
    sns.boxplot(data=hits, x="era", y=col, hue="era",
                order=["album","streaming"], ax=ax,
                palette=["#4C72B0", "#DD8452"], legend=False)
    ax.set_title(col)
    ax.set_xlabel("")
plt.tight_layout()
plt.show()


The duration boxplot makes the change really obvious. Both the box and the whiskers shifted
down. The loudness boxplot is almost as clean. The valence one is more subtle but the median
clearly moved.

## Section 8. Statistical test: bootstrap mean differences

The plots show patterns but I want a number on each one. How confident am I that this isn't just
noise? I'm using a bootstrap test instead of a t-test for two reasons. First, the distributions
aren't normal (look at the boxplots), and a t-test assumes they roughly are. Second, bootstraps
are easier to explain to someone who isn't a statistician, since the procedure is basically just
"resample the data a bunch of times and look at the spread of the answers."

The procedure: resample both eras with replacement 2000 times, compute the difference in means
each time, and report the central 95 percent of those differences as the confidence interval. The
p-value is the share of bootstrap differences on the opposite side of zero from the point
estimate, doubled (two-sided). This bootstrap approach is the project's "feature not covered in
class" item from the rubric. We covered classical t-tests and z-tests but not resampling-based
inference, and bootstrap is exactly the kind of tool that lets you put confidence intervals on a
non-normal-looking metric without having to wave your hands at the assumptions.


In [ ]:
def bootstrap_diff(a, b, n_boot=2000, seed=42):
    """
    Bootstrap CI for (mean of a) - (mean of b).
    Returns: (point estimate, 2.5%ile, 97.5%ile, two-sided p-value)
    Method described in Efron and Hastie's Computer Age Statistical Inference.
    """
    rng = np.random.default_rng(seed)
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    a = a[~np.isnan(a)]
    b = b[~np.isnan(b)]
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        sa = rng.choice(a, size=len(a), replace=True)
        sb = rng.choice(b, size=len(b), replace=True)
        diffs[i] = sa.mean() - sb.mean()
    point = a.mean() - b.mean()
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    p = 2 * np.mean(diffs * np.sign(point) < 0)
    return point, lo, hi, p


album = hits[hits["era"] == "album"]
stream = hits[hits["era"] == "streaming"]

results = []
for f in features + ["explicit"]:
    point, lo, hi, p = bootstrap_diff(stream[f].values, album[f].values)
    results.append({
        "feature": f,
        "album_mean": float(np.nanmean(album[f])),
        "stream_mean": float(np.nanmean(stream[f])),
        "diff (stream - album)": point,
        "ci_lo": lo,
        "ci_hi": hi,
        "p_value": p,
        "significant": p < 0.05,
    })

res_df = pd.DataFrame(results)
res_df.round(4)


Reading the table:

duration_s. The difference is about negative 35.7 seconds (CI from negative 39 to negative 32,
p near zero). Streaming-era hits are about half a minute shorter on average. This is the
strongest single result in the project.

loudness. The difference is about positive 1.49 dB (CI 1.31 to 1.68, p near zero). Hits got
significantly louder.

explicit. The difference is about positive 16 percentage points (CI 13 to 20, p near zero). The
explicit-content rate roughly doubled, from 17 percent to 34 percent.

valence. The difference is about negative 0.061 (CI from negative 0.079 to negative 0.044, p
near zero). Hits sound less positive. The "music got sadder" claim has support.

tempo. The difference is about positive 3.3 BPM (p around 0.004). Real but small.

energy. The difference is about positive 0.014 (p around 0.025). Borderline significant, very
small effect.

acousticness. The difference is about negative 0.012 (CI from negative 0.028 to positive 0.003,
p around 0.12). Not significant. The "less acoustic" claim doesn't hold up at the 5 percent
level.

danceability. The difference is about positive 0.009 (p around 0.13). Not significant.
Streaming-era hits aren't measurably more danceable than album-era hits.

So three hypotheses confirmed strongly, two not confirmed, and one bonus finding about valence
that wasn't part of the original streaming-era story. The "shorter, louder, more explicit" piece
of the journalist narrative is real. The "more danceable, less acoustic" piece isn't.

## Section 9. Genre control: is this just hip-hop's takeover?

A real concern with all of the above: maybe the era effect is fake, and what's actually happening
is that hip-hop (which has shorter, louder tracks) became more commercially dominant after 2010.
That would be a textbook Simpson's paradox. The overall change would be driven by genre mix, not
by genre behavior changing.

To check, I pull each hit's genres from the artist_genres bridge table, group them under a small
set of umbrella genres, and ask: did the within-genre medians shift the same way? If yes, the
era effect is robust. If only hip-hop moved and the others didn't, the result would mostly be a
chart-composition story.


In [ ]:
sql_genre = '''
SELECT bb.bb_year AS year, bb.bb_rank AS rank, g.name AS genre
FROM billboard_hits bb
JOIN track_artists ta ON ta.track_id = bb.track_id
JOIN artist_genres ag ON ag.artist_id = ta.artist_id
JOIN genres g         ON g.genre_id = ag.genre_id
WHERE bb.track_id IS NOT NULL
'''
genre_long = pd.read_sql(sql_genre, conn)

# spotify has 5000+ niche genre tags ('chillwave indietronica' etc).
# I'm rolling them up into 6 umbrellas so I can actually plot them.
def umbrella(g):
    g = g.lower()
    if "hip hop" in g or "rap" in g or "trap" in g: return "hip-hop/rap"
    if "country" in g: return "country"
    if "r&b" in g or "soul" in g: return "r&b/soul"
    if "rock" in g or "metal" in g or "punk" in g: return "rock"
    if "pop" in g: return "pop"
    if "electronic" in g or "edm" in g or "house" in g or "techno" in g or "dance" in g:
        return "electronic/dance"
    return None

genre_long["umbrella"] = genre_long["genre"].apply(umbrella)
genre_long = genre_long[genre_long["umbrella"].notna()]

# pick the most common umbrella for each (year, rank) pair
primary = (genre_long.groupby(["year","rank","umbrella"]).size()
           .reset_index(name="n")
           .sort_values(["year","rank","n"], ascending=[True,True,False])
           .drop_duplicates(["year","rank"])[["year","rank","umbrella"]])

merged = hits.merge(primary, on=["year","rank"], how="left")
print("hits with a primary genre:", merged["umbrella"].notna().sum())
print()
print("Median duration_s by era and primary genre:")
dur_by_genre = merged.groupby(["umbrella","era"])["duration_s"].median().unstack()
print(dur_by_genre.round(1))


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
dur_by_genre.dropna().plot(kind="bar", ax=ax, color=["#4C72B0", "#DD8452"])
ax.set_ylabel("median duration (seconds)")
ax.set_title("Did songs get shorter in every genre? (yes)")
ax.set_xlabel("primary genre")
ax.legend(title="era")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


This is the key plot for ruling out Simpson's paradox. Every umbrella genre shows shorter
streaming-era hits. Rock dropped about 52 seconds, country dropped 21, pop dropped 23. The
shrinkage isn't a hip-hop artifact, it's a real cross-genre pattern. R&B and soul barely shrunk
and is the one exception worth flagging.

## Section 10. Correlation heatmap among streaming-era hits

One more piece, mostly as a gut check. Which audio features are actually independent of each
other? Energy and loudness should be tightly correlated, since loud music is usually
high-energy. Acousticness should be inversely correlated with energy (acoustic music tends to be
calmer). Valence should correlate weakly with energy (happy songs lean energetic). If those
expectations don't show up in the data, my whole analysis is suspect.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
corr = stream[features].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0,
            square=True, ax=ax, cbar_kws={"shrink": 0.8})
ax.set_title("Audio feature correlations among streaming-era hits")
plt.tight_layout()
plt.show()


Energy and loudness: 0.74 (very strong, as expected). Energy and acousticness: negative 0.49 (also
expected). Valence and energy: 0.44 (mild correlation, happy songs lean energetic). Tempo doesn't
correlate strongly with anything else, which is a useful reminder that "fast" and "energetic"
aren't the same thing in audio terms.

Nothing here is surprising, but it's a useful gut-check that Spotify's audio features behave the
way I'd expect them to. If the correlations were weird, that would be a sign that something is
wrong with the data load or that the features themselves don't mean what I think they do.

## Conclusions

So, did streaming change what a hit sounds like? Mostly yes, but in ways that are more specific
than the headline version of the story.

Hits got dramatically shorter. Median duration dropped from 4 minutes 10 seconds to 3 minutes 39
seconds, which is a 30-second reduction, statistically significant at any reasonable level (95
percent CI from negative 39 to negative 32 seconds), and crucially, it happened in every major
genre I looked at. Rock songs dropped by almost a full minute. This is the clearest support for
the streaming-era thesis.

Hits got louder. Median loudness rose by about 1.5 dB. This trend predates streaming (the
"loudness wars" started in the late 90s) but the data shows it kept marching forward.

Explicit content roughly doubled. From 17 percent of album-era hits to 34 percent of streaming-era
hits. This is consistent with streaming letting labels bypass radio's content gates.

Hits got slightly less happy (lower valence). A surprise finding, not part of the original
streaming-era hypothesis but it's clearly there in the data, with a confident confidence interval.

The "more danceable, less acoustic" claim doesn't hold up. Both differences trended in the
expected direction but neither was statistically significant. The popular framing of the streaming
era as making music "more dancey," at least at the level of Billboard hits, isn't really
supported here.

The shrinkage isn't just hip-hop's takeover. Within-genre comparisons show the pattern in
country, rock, pop, R&B, and electronic music too. So this is a real change in how songs are
written and produced, not a Simpson's-paradox artifact of which genres charted.

What this can't tell us: whether streaming caused these changes or just coincided with them. 2010
is also when smartphones went mainstream, when YouTube became a primary discovery channel, and
when the album as an art form started really losing share. Any of those could plausibly drive
shorter, louder, more attention-grabbing songwriting. The data here can rule out a few
explanations (it's not just hip-hop, the change is real) but it can't pick a single cause.

For a follow-up I'd want to see whether the first 30 seconds of streaming-era songs sound
systematically different from the first 30 seconds of album-era songs. That's the most specific
version of the "engineered to beat the skip threshold" hypothesis, and Spotify's public audio
analysis API would let me check it directly.


In [ ]:
# Fin
conn.close()
print("done")
